# Baseline per la predizione di `delta_rtt`

*Progetto: tokenizzazione dei network flow CERN (dati Scitags Firefly), Gruppo 1.*

**Scopo.** Fissare una scala di riferimento per valutare la tokenizzazione: dal *pavimento*
(nessuna informazione) al *soffitto pratico* (modelli forti su input non tokenizzati).
Il compito è prevedere `delta_rtt = end_rtt − start_rtt` (ms) al momento dell'inizio del flow.

**Dati.** `flows_prepared.parquet`: flow con `start_rtt > 0` e `end_rtt > 0`, split cronologico
70/15/15 (train fino al 12/10 17:12, validation fino al 13/10 22:18, poi test).

**Regole.**
- Tutto ciò che si stima (mediane, soglie, vocabolari, parametri) si stima **solo sul train**.
- Si confrontano i modelli sulla **validation**; il **test** si usa una sola volta, alla fine del progetto.
- Vietati come input i campi noti solo alla fine del flow: `end_rtt`, durata, colonne `an_*`
  (usate solo per analizzare gli errori).
- Metrica principale: **MAE**; accanto RMSE, RMSE robusto †, P90 e P99 dell'errore assoluto.

**Struttura.** §1 Setup · §2 Valutazione · §3 B0 mediana globale · §4 B1 mediana per tratta ·
§5 Preparazione per i modelli · §6 B4 LightGBM · §7 B2 ridge · §8 B3 rete neurale · §9 Conclusioni.

## 1. Setup

Import, percorsi dei dati, connessione a DuckDB e vista `fp` sul file preparato.
DuckDB interroga il file Parquet direttamente dal disco: in Python arrivano solo i risultati
delle query, non i 67 M di righe.

In [8]:
import json
import sys
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# --- Percorsi ---
PROJECT_ROOT = Path.cwd().parent                     # il notebook sta in results/
DATA_DIR = PROJECT_ROOT / "data" / "student_dataset"
FLOWS    = DATA_DIR / "flows.parquet"
SPLITS   = DATA_DIR / "splits.json"
PREPARED = DATA_DIR / "flows_prepared.parquet"

for f in [FLOWS, SPLITS, PREPARED]:
    print(f"{f.name:25s} esiste: {f.exists()}")

# --- Connessione DuckDB ---
con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")
con.execute(f"SET temp_directory = '{(PROJECT_ROOT / 'duckdb_tmp').as_posix()}'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET TimeZone = 'UTC'")

con.execute(f"CREATE OR REPLACE VIEW fp AS SELECT * FROM read_parquet('{PREPARED.as_posix()}')")

print("Python", sys.version.split()[0], "| DuckDB", duckdb.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

flows.parquet             esiste: True
splits.json               esiste: True
flows_prepared.parquet    esiste: True
Python 3.14.0 | DuckDB 2.0.0 | pandas 2.3.3 | numpy 2.5.3


### Controllo dei dati

Righe, inizio e fine di ogni split (attesi: train 46.730.686, validation 10.069.254,
test 10.086.352) e numero di colonne del file preparato (attese 41).
Per il test si leggono solo questi conteggi descrittivi, nessun valore del target.

In [9]:
display(con.execute("""
    SELECT split,
           COUNT(*)     AS n,
           MIN(t_start) AS inizio,
           MAX(t_start) AS fine
    FROM fp
    GROUP BY split
    ORDER BY MIN(t_start)
""").df())

colonne = con.execute("DESCRIBE fp").df()
print(f"Colonne: {len(colonne)}")

,split,n,inizio,fine
0,train,46730686,2026-10-07 23:59:59.465736+00:00,2026-10-12 17:12:05.477960+00:00
1,val,10069254,2026-10-12 17:12:05.481692+00:00,2026-10-13 22:18:28.700870+00:00
2,test,10086352,2026-10-13 22:18:28.701549+00:00,2026-10-14 23:59:59.753340+00:00


Colonne: 41


## 2. Valutazione

Tutti i modelli si valutano sulla validation con la stessa funzione e le stesse metriche
(errore $e_i = y_i - \hat y_i$, in ms):

| Metrica | Definizione | Lettura |
|---|---|---|
| **MAE** (principale) | media di $\lvert e_i \rvert$ | errore tipico |
| RMSE | $\sqrt{\text{media di } e_i^2}$ | dominato dagli errori grandi |
| RMSE robusto † | RMSE sui soli flow *tipici* | errori grandi sui flow normali |
| P90, P99 | 90° e 99° percentile di $\lvert e_i \rvert$ | "nel 90 % dei flow sbaglio meno di…" |

### 2.1 Flow tipici (per l'RMSE robusto)

Sulla B0, lo 0,1 % dei flow produce il 98,8 % dell'errore quadratico: l'RMSE completo misura quasi
solo quei pochi casi. Definiamo quindi un insieme **fisso** di flow *tipici*: quelli con `delta_rtt`
tra il quantile 0,05 % e il 99,95 % **del train**. L'insieme dipende dal valore vero, non dalle
previsioni, quindi è lo stesso per tutti i modelli. Ogni tabella riporta sempre entrambi gli RMSE.

In [10]:
Q_LO, Q_HI = 0.0005, 0.9995

LO, HI = con.execute(f"""
    SELECT quantile_cont(delta_rtt, [{Q_LO}, {Q_HI}])
    FROM fp
    WHERE split = 'train'
""").fetchone()[0]

NOTA_ROBUSTO = (f"† RMSE robusto: solo flow con delta_rtt in [{LO:.2f}, {HI:.2f}] ms, "
                f"quantili {Q_LO:.2%}–{Q_HI:.2%} del train (insieme fisso, uguale per tutti i modelli).")

print(f"LO = {LO:.3f} ms | HI = {HI:.3f} ms")

con.execute(f"""
    SELECT split,
           COUNT(*)                                                       AS n,
           COUNT(*) FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})        AS n_tipici,
           100 * AVG(CASE WHEN delta_rtt BETWEEN {LO} AND {HI} THEN 1 ELSE 0 END) AS pct_tipici
    FROM fp
    WHERE split IN ('train', 'val')
    GROUP BY split
    ORDER BY split
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LO = -184.233 ms | HI = 58.823 ms


,split,n,n_tipici,pct_tipici
0,train,46730686,46683954,99.899997
1,val,10069254,10061030,99.918326


### 2.2 Funzione di valutazione

`valuta(vista)` riceve il nome di una tabella o vista DuckDB con le righe della validation,
la colonna vera `delta_rtt`, la previsione `pred` e le colonne dei sottogruppi.
Calcola le metriche in DuckDB (in Python arrivano solo i numeri), globalmente e per sottogruppo:

| Sottogruppo | Definizione | Perché |
|---|---|---|
| globale | tutti i flow | metrica ufficiale |
| FNAL | server o client = `USCMS-FNAL-WC1` | target del train alterato da un guasto di telemetria (in indagine) |
| sito mancante | server o client = `MISSING` | ~18 % dei flow senza sito CRIC |
| verso incerto | `verso_incerto` | ruoli server/client assegnati in modo incerto (0,27 %) |
| flow > 1 min | `an_durata_s > 60` | origine del picco a ~5 ms; durata usata **solo per dividere** |

`mostra(ris)` stampa la tabella con nomi leggibili e la nota sull'RMSE robusto.

In [13]:
SOTTOGRUPPI = {
    "globale":       "TRUE",
    "FNAL":          "server_site = 'USCMS-FNAL-WC1' OR client_site = 'USCMS-FNAL-WC1'",
    "sito mancante": "server_site = 'MISSING' OR client_site = 'MISSING'",
    "verso incerto": "verso_incerto",
    "flow > 1 min":  "an_durata_s > 60",
}

def valuta(vista):
    """vista: tabella/vista DuckDB con delta_rtt, pred e le colonne dei sottogruppi (solo validation)."""
    parti = []
    for ordine, (nome, cond) in enumerate(SOTTOGRUPPI.items()):
        parti.append(f"""
            SELECT {ordine} AS ordine, '{nome}' AS gruppo, COUNT(*) AS n,
                   AVG(ABS(delta_rtt - pred))                            AS mae,
                   SQRT(AVG(POWER(delta_rtt - pred, 2)))                 AS rmse,
                   SQRT(AVG(POWER(delta_rtt - pred, 2))
                        FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})) AS rmse_robusto,
                   quantile_cont(ABS(delta_rtt - pred), 0.90)            AS p90_abs,
                   quantile_cont(ABS(delta_rtt - pred), 0.99)            AS p99_abs
            FROM {vista}
            WHERE {cond}
        """)
    query = "SELECT * FROM (" + " UNION ALL ".join(parti) + ") ORDER BY ordine"
    return con.execute(query).df().drop(columns="ordine")

NOMI_METRICHE = {"mae": "MAE", "rmse": "RMSE", "rmse_robusto": "RMSE robusto †",
                 "p90_abs": "P90 |err|", "p99_abs": "P99 |err|"}

def mostra(ris):
    """Mostra la tabella delle metriche (ms) con nomi leggibili e la nota sull'RMSE robusto."""
    display(ris.rename(columns=NOMI_METRICHE).round(3))
    print(NOTA_ROBUSTO)

### 2.3 Tabella dei risultati

`RESULTS` raccoglie una riga per modello: metriche globali sulla validation, numero di parametri
(o di valori memorizzati) e tempo di **solo training** (la valutazione è esclusa).
`registra` aggiunge un modello; se il modello è già presente, la riga viene sostituita, non duplicata.

In [11]:
RESULTS = []

def registra(modello, ris, parametri, tempo_train_s):
    """Aggiunge (o sostituisce) la riga di un modello in RESULTS usando la riga 'globale' di ris."""
    global RESULTS
    glob = ris.set_index("gruppo").loc["globale"]
    RESULTS = [r for r in RESULTS if r["modello"] != modello]
    RESULTS.append({
        "modello":        modello,
        "MAE":            glob["mae"],
        "RMSE":           glob["rmse"],
        "RMSE robusto †": glob["rmse_robusto"],
        "P90 |err|":      glob["p90_abs"],
        "P99 |err|":      glob["p99_abs"],
        "parametri":      parametri,
        "tempo_train_s":  round(tempo_train_s, 1),
    })

def tabella_risultati():
    """Mostra RESULTS come tabella, con la nota sull'RMSE robusto."""
    display(pd.DataFrame(RESULTS).round(3))
    print(NOTA_ROBUSTO)

## 3. Baseline 0 – Mediana globale

**Idea.** Prevedere per ogni flow la stessa costante, senza usare alcun input. È il *pavimento*:
qualunque modello deve fare meglio.

**Perché la mediana.** Tra tutte le costanti $c$, quella che minimizza
$\frac{1}{n}\sum_i \lvert y_i - c \rvert$ (la MAE) è la mediana; quella che minimizza l'errore
quadratico è la media. Calcoliamo entrambe per confronto.

**Come.** Mediana e media sul **train completo** (46,7 M flow, calcolate da DuckDB senza caricarle
in memoria: nessun rumore di campionamento). Previsione = mediana per ogni flow della validation.
Il tempo di training è solo quello del calcolo della mediana.

In [14]:
# --- "Training": mediana e media del train ---
t0 = time.time()
mediana, media = con.execute("""
    SELECT MEDIAN(delta_rtt), AVG(delta_rtt)
    FROM fp
    WHERE split = 'train'
""").fetchone()
t_train = time.time() - t0
print(f"train → mediana: {mediana:.3f} ms | media: {media:.3f} ms | tempo: {t_train:.1f} s")

# --- Previsione sulla validation ---
con.execute(f"""
    CREATE OR REPLACE VIEW pred_b0 AS
    SELECT delta_rtt, server_site, client_site, verso_incerto, an_durata_s,
           {mediana} AS pred
    FROM fp
    WHERE split = 'val'
""")

# --- Valutazione e registrazione ---
ris_b0 = valuta("pred_b0")
registra("B0 mediana globale", ris_b0, parametri=1, tempo_train_s=t_train)
mostra(ris_b0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train → mediana: 4.398 ms | media: 3.172 ms | tempo: 3.0 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gruppo,n,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|
0,globale,10069254,3.298,47.289,5.812,4.510,23.552
1,FNAL,215471,1.636,3.193,2.666,4.381,6.279
2,sito mancante,2122477,3.539,16.574,6.885,4.847,25.834
3,verso incerto,32662,4.570,8.801,4.790,4.423,5.541
4,flow > 1 min,6632723,3.129,58.019,6.453,4.643,27.535


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).


## 4. Baseline 1 – Mediana per tratta, con fallback gerarchico

**Idea.** Una *tratta* è la coppia di estremi di un flow (dove sta il server, dove sta il client).
Per ogni tratta calcoliamo la mediana di `delta_rtt` sul train completo; per ogni flow della validation
prevediamo la mediana della sua tratta. È una tabella di consultazione, senza parametri ottimizzati:
misura quanto spiega il solo "dove". In termini matematici stima la mediana condizionata
$\text{med}(Y \mid \text{tratta})$, cioè una funzione costante a tratti.

**Varianti (6) = 2 modi di identificare un estremo × 3 modi di ordinare la coppia.**

| | non orientata {A, B} | server → client | src → dst |
|---|---|---|---|
| siti CRIC (`MISSING` incluso) | B1 site non_orientata | B1 site server_client | B1 site src_dst |
| reti IP L2 (/24, /48) | B1 L2 non_orientata | B1 L2 server_client | B1 L2 src_dst |

- **Non orientata:** (A, B) e (B, A) sono la stessa tratta.
- **Server → client:** conta chi offre il servizio e chi lo richiede (ruolo).
- **Src → dst:** conta l'ordine con cui il monitoraggio ha registrato il flow (dipende dal software).

**Fallback (uguale per tutte le varianti).** Una tratta si usa solo se ha almeno **N = 1.000** flow
nel train, altrimenti la sua mediana è troppo rumorosa. In quel caso si usa la mediana della coppia
di operatori di rete (ASN, stesso orientamento); se anche questa ha meno di N flow, la mediana globale (B0).

**Domande.** La direzione conta? Se sì, conta il ruolo o l'ordine di registrazione?
Le reti IP (sempre presenti) battono i siti (mancanti nel ~18 % dei flow)?

In [16]:
# Espressioni SQL delle due chiavi di una coppia, per orientamento.
# {p} = prefisso di tabella ("f." oppure ""), {x} = identificatore ("site", "L2", "asn")
ORIENTAMENTI = {
    "non_orientata": ("LEAST({p}server_{x}, {p}client_{x})",
                      "GREATEST({p}server_{x}, {p}client_{x})"),
    "server_client": ("{p}server_{x}", "{p}client_{x}"),
    "src_dst":       ("{p}src_{x}",    "{p}dst_{x}"),
}

def chiavi(orient, x, p=""):
    """Le due espressioni SQL della coppia, per un orientamento e un identificatore."""
    k1, k2 = ORIENTAMENTI[orient]
    return k1.format(p=p, x=x), k2.format(p=p, x=x)

def baseline1(ident, orient, N=1000):
    """ident: 'site' o 'L2'; orient: chiave di ORIENTAMENTI; N: flow minimi di train per gruppo."""
    t0 = time.time()
    # Mediane sul train: livello 1 = coppia di siti/reti, livello 2 = coppia di ASN
    for livello, x in [(1, ident), (2, "asn")]:
        k1, k2 = chiavi(orient, x)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE med_l{livello} AS
            SELECT {k1} AS k1, {k2} AS k2,
                   MEDIAN(delta_rtt) AS med,
                   COUNT(*)          AS n
            FROM fp
            WHERE split = 'train'
            GROUP BY ALL
            HAVING COUNT(*) >= {N}
        """)
    t_train = time.time() - t0

    # Previsione sulla validation: vince il primo livello disponibile
    a1, b1 = chiavi(orient, ident, p="f.")
    a2, b2 = chiavi(orient, "asn", p="f.")
    con.execute(f"""
        CREATE OR REPLACE VIEW pred_b1 AS
        SELECT f.delta_rtt, f.server_site, f.client_site, f.verso_incerto, f.an_durata_s,
               COALESCE(m1.med, m2.med, {mediana}) AS pred,
               CASE WHEN m1.med IS NOT NULL THEN '1 coppia {ident}'
                    WHEN m2.med IS NOT NULL THEN '2 coppia ASN'
                    ELSE '3 globale' END AS livello
        FROM fp AS f
        LEFT JOIN med_l1 AS m1 ON {a1} = m1.k1 AND {b1} = m1.k2
        LEFT JOIN med_l2 AS m2 ON {a2} = m2.k1 AND {b2} = m2.k2
        WHERE f.split = 'val'
    """)

    ris = valuta("pred_b1")
    livelli = con.execute("""
        SELECT livello,
               COUNT(*)                                  AS n,
               100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()  AS pct,
               AVG(ABS(delta_rtt - pred))                AS mae
        FROM pred_b1
        GROUP BY livello
        ORDER BY livello
    """).df()
    n_param = sum(con.execute(f"SELECT COUNT(*) FROM med_l{l}").fetchone()[0] for l in (1, 2)) + 1
    return ris, livelli, n_param, t_train

### Esecuzione delle sei varianti

Per ogni combinazione di identificatore e orientamento: mediane sul train, valutazione sulla validation,
registrazione in `RESULTS`. Le tabelle di dettaglio (sottogruppi e livelli del fallback) restano in `DETTAGLI_B1`.

In [17]:
DETTAGLI_B1 = {}

for ident in ["site", "L2"]:
    for orient in ORIENTAMENTI:
        nome = f"B1 {ident} {orient}"
        ris, livelli, n_param, t_train = baseline1(ident, orient)
        DETTAGLI_B1[nome] = {"ris": ris, "livelli": livelli}
        registra(nome, ris, parametri=n_param, tempo_train_s=t_train)
        print(f"{nome:26s} MAE = {ris.loc[ris['gruppo'] == 'globale', 'mae'].iloc[0]:.3f} "
              f"| livello 1: {livelli['pct'].iloc[0]:.1f} % | {t_train:.1f} s")

tabella_risultati()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 site non_orientata      MAE = 2.463 | livello 1: 99.8 % | 4.2 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 site server_client      MAE = 2.270 | livello 1: 99.7 % | 3.6 s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (8 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

B1 L2 src_dst              MAE = 2.310 | livello 1: 97.8 % | 3.9 s


,modello,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|,parametri,tempo_train_s
0,B0 mediana globale,3.298,47.289,5.812,4.510,23.552,1,3.0
1,B1 site non_orientata,2.463,47.252,5.515,4.977,23.283,816,4.2
2,B1 site server_client,2.270,47.241,5.419,4.909,23.188,1077,3.6
3,B1 site src_dst,2.401,47.248,5.480,4.953,23.276,1029,3.3
4,B1 L2 non_orientata,2.338,47.241,5.440,4.884,23.240,1920,6.0
5,B1 L2 server_client,2.209,47.233,5.377,4.861,23.068,2166,4.2
6,B1 L2 src_dst,2.310,47.239,5.423,4.868,23.236,2140,3.9


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).


## 5. Preparazione dei dati per i modelli

I modelli (B2–B4) ricevono ogni flow come una riga di numeri: un **codice intero** per ogni input
categorico e `start_rtt`. Passi:
1. input dei modelli;
2. vocabolari (valore → codice), stimati sul train;
3. tabella `mappa` in DuckDB;
4. query di codifica;
5. insiemi di dati: `train_enc` (addestramento), `tune_enc` (early stopping e scelte), `val_enc` (risultati).

### 5.1 Input dei modelli
Orientamento server → client e reti IP L2 (le scelte migliori nella B1). Input:
- numerico: `start_rtt`;
- categorici: `ip_version`, `experiment`, `exp_act`, `application`, `hostname`,
  `server_site`, `client_site`, `server_L2`, `client_L2`, `porta_servizio`, `porta_client`,
  `verso`, `verso_incerto`.

Siti e reti L2 hanno un **vocabolario condiviso** tra server e client: lo stesso posto riceve lo stesso
codice in entrambi i ruoli. Esclusi tutti i campi noti solo alla fine del flow (durata, `end_rtt`, `an_*`).

In [18]:
FEATURE_NUM = ["start_rtt"]
TARGET = "delta_rtt"

# vocabolario → colonne che lo usano
VOCAB_COLONNE = {
    "ip_version":     ["ip_version"],
    "experiment":     ["experiment"],
    "exp_act":        ["exp_act"],
    "application":    ["application"],
    "hostname":       ["hostname"],
    "site":           ["server_site", "client_site"],   # condiviso
    "L2":             ["server_L2", "client_L2"],       # condiviso
    "porta_servizio": ["porta_servizio"],
    "porta_client":   ["porta_client"],
    "verso":          ["verso"],
    "verso_incerto":  ["verso_incerto"],
}

# elenco piatto di tutte le colonne categoriche
FEATURE_CAT = [c for colonne in VOCAB_COLONNE.values() for c in colonne]

print(f"{len(VOCAB_COLONNE)} vocabolari, {len(FEATURE_CAT)} colonne categoriche:")
print(FEATURE_CAT)

11 vocabolari, 13 colonne categoriche:
['ip_version', 'experiment', 'exp_act', 'application', 'hostname', 'server_site', 'client_site', 'server_L2', 'client_L2', 'porta_servizio', 'porta_client', 'verso', 'verso_incerto']


### 5.2 Vocabolari (stimati sul train)

Codici riservati: `0 = PAD` (per i modelli a token, qui inutilizzato), `1 = MISSING`,
`2 = RARE` (visto nel train ma con meno di N = 1.000 flow), `3 = UNK` (mai visto nel train).
I valori con almeno N flow ricevono i codici da 4 in su, in ordine di frequenza.
Per i vocabolari condivisi si contano insieme le occorrenze nei due ruoli.
I valori sono convertiti in testo, così numeri (`ip_version`) e vero/falso (`verso_incerto`)
si trattano come le altre categorie. I vocabolari usano solo i conteggi dei valori, non il target.

In [20]:
N_MIN = 1000
PAD, MISSING, RARE, UNK = 0, 1, 2, 3

VOCAB = {}   # nome vocabolario → {valore: codice} dei valori frequenti
VISTI = {}   # nome vocabolario → insieme di tutti i valori visti nel train

t0 = time.time()
for nome, colonne in VOCAB_COLONNE.items():
    # una SELECT per colonna, impilate una sotto l'altra con UNION ALL
    parti = [f"SELECT CAST({c} AS VARCHAR) AS valore FROM fp WHERE split = 'train'"
             for c in colonne]
    conteggi = con.execute(f"""
        SELECT valore, COUNT(*) AS n
        FROM ({" UNION ALL ".join(parti)})
        GROUP BY valore
        ORDER BY n DESC
    """).df()

    frequenti = conteggi.loc[(conteggi["n"] >= N_MIN) & (conteggi["valore"] != "MISSING"), "valore"]
    VOCAB[nome] = {v: i + 4 for i, v in enumerate(frequenti)}
    VISTI[nome] = set(conteggi["valore"])

    print(f"{nome:15s} valori nel train: {len(conteggi):6d} | con ≥ {N_MIN} flow: {len(VOCAB[nome]):5d}")

print(f"Tempo: {time.time() - t0:.1f} s")

ip_version      valori nel train:      2 | con ≥ 1000 flow:     2
experiment      valori nel train:      6 | con ≥ 1000 flow:     6
exp_act         valori nel train:     31 | con ≥ 1000 flow:    29
application     valori nel train:    168 | con ≥ 1000 flow:    56
hostname        valori nel train:    749 | con ≥ 1000 flow:   686
site            valori nel train:     91 | con ≥ 1000 flow:    83
L2              valori nel train:    720 | con ≥ 1000 flow:   340
porta_servizio  valori nel train:     47 | con ≥ 1000 flow:    46
porta_client    valori nel train:      4 | con ≥ 1000 flow:     4
verso           valori nel train:      2 | con ≥ 1000 flow:     2
verso_incerto   valori nel train:      2 | con ≥ 1000 flow:     2
Tempo: 4.6 s


### 5.3 Tabella `mappa`: valore → codice

Per ogni vocabolario e ogni valore visto nel train: il codice se frequente, `1` se `MISSING`,
`2` (RARE) se presente ma con meno di N flow. Un valore assente dalla tabella (mai visto nel train)
riceverà `3` (UNK). La tabella è salvata in DuckDB, così la codifica di milioni di righe avviene lì
e in Python arrivano solo interi.

In [21]:
righe = []
for nome in VOCAB_COLONNE:
    for valore in VISTI[nome]:
        if valore == "MISSING":
            codice = MISSING
        elif valore in VOCAB[nome]:
            codice = VOCAB[nome][valore]
        else:
            codice = RARE
        righe.append((nome, valore, codice))

mappa_df = pd.DataFrame(righe, columns=["nome", "valore", "codice"])
con.register("mappa_df", mappa_df)
con.execute("CREATE OR REPLACE TABLE mappa AS SELECT * FROM mappa_df")

print(f"Righe nella mappa: {len(mappa_df):,}")
mappa_df.groupby("nome")["codice"].agg(
    valori="count",
    rari=lambda s: (s == RARE).sum(),
)

Righe nella mappa: 1,822


,valori,rari
nome,,
L2,720,380
application,168,111
exp_act,31,2
experiment,6,0
hostname,749,63
ip_version,2,0
porta_client,4,0
porta_servizio,47,1
site,91,7


## Stato del lavoro (17 novembre 2026)

**Fatto e verificato**
- **§1 Setup.** Connessione DuckDB e vista `fp` su `flows_prepared.parquet`; conteggi per split corretti
  (train 46.730.686 · val 10.069.254 · test 10.086.352), 41 colonne.
- **§2 Valutazione.** Soglie del set tipico dal train (LO = −184,23 ms, HI = 58,82 ms; 99,90 % del train,
  99,92 % della validation); funzione `valuta` (MAE, RMSE, RMSE robusto †, P90, P99, globale e per
  sottogruppo); tabella `RESULTS` con `registra`.
- **§3 B0 – mediana globale.** Mediana del train 4,398 ms → MAE 3,298 ms (pavimento).
- **§4 B1 – mediana per tratta.** Sei varianti; la migliore è reti IP L2 + server → client:
  MAE 2,209 ms (−33 %). Conta il ruolo server/client; le reti IP battono i siti. FNAL peggiora per un
  drift tra train e validation (indagine e decisione nel notebook `03_baseline_diario`: dati invariati).
  Tutti i risultati coincidono con il notebook precedente.
- **§5 Preparazione per i modelli (in corso).**
  - 5.1 Input: 13 categorici (server/client, reti L2) + `start_rtt`.
  - 5.2 Vocabolari sul train (N = 1.000): ad es. `L2` 720 valori → 340 frequenti, `hostname` 749 → 686,
    `application` 168 → 56, `site` 91 → 83 (escluso `MISSING`).
  - 5.3 Tabella `mappa` in DuckDB: 1.822 coppie valore → codice.

**Decisioni prese**
- Loss L1 (coerente con la MAE) per LightGBM, modello lineare e rete neurale.
- Early stopping e scelta delle impostazioni sull'**ultimo 10 % del periodo di train** (`tune_enc`);
  la validation resta solo per i risultati.
- B2: modello lineare addestrato con errore assoluto (PyTorch).
- B3: embedding per campo (da confermare con il gruppo della tokenizzazione).

**Prossimi passi**
1. 5.4 Query di codifica e 5.5 insiemi `train_enc` (~3 M), `tune_enc` (~1 M), `val_enc` (10,07 M).
2. Controlli di qualità (quote RARE/UNK per campo) e salvataggio dei vocabolari in JSON per il gruppo.
3. §6 B4 LightGBM (+ diagnostica con la durata) · §7 B2 lineare L1 · §8 B3 rete neurale · §9 Conclusioni.